In [ ]:
from glob import glob
import subprocess
import time
from Bio import SeqIO
from Bio.Seq import Seq
import os.path

In [ ]:
read_len = 70
overlap = read_len // 2
wd = '.'
wd += '/'

In [ ]:
def fetch_ids():
    iddict = dict()
    for filename in glob(f'{wd}clusters/*'):
        protein = filename.split(f'{wd}clusters/joined_max_clique_')[-1].split('.')[0]
        with open(filename, 'r') as inp:
            iddict[protein] = [l.split('(refseq:')[1].split(')')[0] for l in inp.readlines() if l.startswith('>')]
    return iddict

In [ ]:
def fetch_datasets(somedict):
    for k in somedict:
        if len(somedict[k]) > 0:
            if os.path.exists(f'{wd}datasets/{k}.zip'):
                print(k)
                with open(f'{wd}temp.txt', 'w') as out:
                    out.write('\n'.join(somedict[k]))
                cmd = f'datasets download gene accession --inputfile {wd}temp.txt --include gene --no-progressbar --filename ./datasets/{k}.zip'
                subprocess.run(cmd, shell=True, check=True)
                time.sleep(61)

In [ ]:
# def fetch_datasets(somedict):
#     query = set()
#     for k in somedict:
#         if len(somedict[k]) > 0:
#             query.add(somedict[k])
#     with open('temp.txt', 'w') as out:
#         out.write('\n'.join(sorted(list(query))))
#     cmd = f'datasets download gene accession --inputfile temp.txt --include gene --no-progressbar --filename ./datasets.zip'
#     subprocess.run(cmd, shell=True, check=True)

In [ ]:
def zip_work(somedict):
    for k in somedict:
        if not os.path.exists(f'{wd}datasets/{k}'):
            cmd = f'mkdir {wd}datasets/{k} && mv {wd}datasets/{k}.zip {wd}datasets/{k} && unzip {wd}datasets/{k}/{k}.zip -d {wd}datasets/{k}'
            subprocess.run(cmd, shell=True, check=True)

In [ ]:
def get_reference_gene(somedict):
    ref_dict = dict()
    for k in somedict:
        if os.path.exists(f'{wd}datasets/{k}/ncbi_dataset/data/gene.fna'):
            with open(f'{wd}datasets/{k}/ncbi_dataset/data/gene.fna', 'r') as inp:
                gene_names = [l for l in inp.readlines() if l.startswith('>')]
            human_genes = dict()
            for g in gene_names:
                if '[organism=Homo sapiens]' in g:
                    human_genes[g[1:].strip()] = len(g.split('.')[0].replace('0', '').split('_')[-1])
            ref_dict[k] = min(human_genes, key=human_genes.get)
    return ref_dict

In [ ]:
def read_fasta(somedict):
    fasta_dict = dict()
    for k in somedict:
        fasta_dict[k] = dict()
        if os.path.exists(f'{wd}datasets/{k}/ncbi_dataset/data/gene.fna'):
            with open(f'{wd}datasets/{k}/ncbi_dataset/data/gene.fna', 'r') as inp:
                fasta_lines = inp.readlines()
            for l in fasta_lines:
                if l.startswith('>'):
                    name = l[1:].strip()
                    fasta_dict[k][name] = ''
                else:
                    fasta_dict[k][name] += l.strip()
    return fasta_dict

In [ ]:
def write_references(fasta_dict, ref_dict):
    for k in ref_dict:
        with open(f'{wd}references/{k}.fasta', 'w') as out:
            out.write(f'>{ref_dict[k].split()[0]}\n{fasta_dict[k][ref_dict[k]]}')

In [ ]:
def delete_not_dna(fasta_dict):
    for k in fasta_dict:
        todel = set()

        for name in fasta_dict[k]:
            if (set(fasta_dict[k][name]) - set('YVKNCWMSARGTDHB')):
                print(name + ' is not a DNA sequence (' + ''.join(set(fasta_dict[k][name])) + '). Omitted')
                todel.add(name)

        for name in todel:
            del fasta_dict[k][name]

    return fasta_dict

In [ ]:
def do_pseudoreads(fasta_dict, read_len, overlap):
    for k in fasta_dict:
        if len(fasta_dict[k]) > 1:
            with open(f'{wd}pseudoreads/{k}.prefastq', 'w') as out:
                for name in fasta_dict[k]:
                    start_str = fasta_dict[k][name]
                    i = 0
                    while len(start_str) > read_len:
                        read = start_str[0:read_len]
                        start_str = start_str[overlap:]
                        out.write('>' + name + '_' + str(i) + '\n' + read + '\n')
                        i += 1
                    out.write('>' + name + '_' + str(i) + '\n' + start_str + '\n')

In [ ]:
def do_fastq(fasta_dict):
    for k in fasta_dict:
        fa_path = f'{wd}pseudoreads/{k}.prefastq'
        fq_path = f'{wd}fastq/{k}.fastq'
        if os.path.exists(fa_path):
            with open(fa_path, "r") as fasta, open(fq_path, "w") as fastq:
                for record in SeqIO.parse(fasta, "fasta"):
                    record.letter_annotations["phred_quality"] = [30] * len(record)
                    SeqIO.write(record, fastq, "fastq")

In [ ]:
iddict = fetch_ids()

fetch_datasets(iddict)

zip_work(iddict)

ref = get_reference_gene(iddict)

fasta = read_fasta(iddict)

write_references(fasta, ref)

fasta = delete_not_dna(fasta)

do_pseudoreads(fasta, read_len, overlap)

do_fastq(fasta)

In [ ]:
%%bash

cd "{wd}"

for ref in references/*.fasta
do
    base=$(basename -- $ref | cut -d . -f 1)
    bwa index "${ref}"
    bwa mem "${ref}" "fastq/${base}.fastq" > "sams/${base}.sam"
    samtools view -bS "sams/${base}.sam" > "sams/${base}.bam"
    samtools sort "sams/${base}.bam" > "sorted/${base}.bam"
    samtools index "sorted/${base}.bam"
    samtools mpileup -f "${ref}" "sorted/${base}.bam" > "pileups/${base}.pileup"
    varscan mpileup2cns "pileups/${base}.pileup" --min-reads2 2 --output-vcf 1 --variants > "vcfs/${base}.vcf"
    rm "sams/${base}.sam"
    rm "sams/${base}.bam"
    rm "pileups/${base}.pileup"
done

In [ ]:
def read_transform_vcf(filename, new_lines, chromosome, minus, start):
    with open(filename, 'r') as inp:
        lines = inp.readlines()

    for l in lines:
        if not (l.startswith('#')):
            if minus:
                pos = str(start - int(l.split('\t')[1]))
            else:
                pos = str(start + int(l.split('\t')[1]))
            l = chromosome + '\t' + pos + '\t' + '\t'.join(l.split('\t')[2:])
            # ref = l.split('\t')[2]
            # alt = l.split('\t')[-1].strip()
            # if '-' in alt:
            #     ref = ref + alt[1:]
            #     alt = ref[0]
            # elif '+' in alt:
            #     alt = ref + alt[1:]
            # if minus:
            #     ref = ref.replace('A', '1').replace('T', '2').replace('G', '3').replace('C', '4')
            #     ref = ref.replace('1', 'T').replace('2', 'A').replace('3', 'C').replace('4', 'G')
            #     alt = alt.replace('A', '1').replace('T', '2').replace('G', '3').replace('C', '4')
            #     alt = alt.replace('1', 'T').replace('2', 'A').replace('3', 'C').replace('4', 'G')
        new_lines.append(l)
    return new_lines

In [ ]:
grch38_chromosomes = [
    "NC_000001.11",
    "NC_000002.12",
    "NC_000003.12",
    "NC_000004.12",
    "NC_000005.10",
    "NC_000006.12",
    "NC_000007.14",
    "NC_000008.11",
    "NC_000009.12",
    "NC_000010.11",
    "NC_000011.10",
    "NC_000012.12",
    "NC_000013.11",
    "NC_000014.9",
    "NC_000015.10",
    "NC_000016.10",
    "NC_000017.11",
    "NC_000018.10",
    "NC_000019.10",
    "NC_000020.11",
    "NC_000021.9",
    "NC_000022.11",
    "NC_000023.11",
    "NC_000024.10"
]

In [ ]:
for vcf in glob(f'{wd}vcfs/*'):
    base = vcf[:-4].split('/')[-1]
    new_lines = list()
    with open(f'{wd}references/{base}.fasta', 'r') as inp:
        first_line = inp.readline()
        if first_line.split(':')[0][1:] in grch38_chromosomes:
            name = first_line.strip().split(':')[1]
            prechromosome = first_line.split('.')[0].split('_')[1]
            stop = False
            i = -1
            while not stop:
                i += 1
                if prechromosome[i] != '0':
                    stop = True
            chromosome = 'chr' + prechromosome[i:]

            if name[0] == 'c':
                minus = True
                name = name[1:]
                start = int(name.split('-')[0]) + 1
            else:
                minus = False
                start = int(name.split('-')[0]) - 1

            new_lines = read_transform_vcf(vcf, new_lines, chromosome, minus, start)

            with open(f'{wd}vcfs_redacted/{base}.vcf', 'w') as out:
                out.write(''.join(new_lines))

In [ ]:
with open(f'{wd}final.vcf', 'w') as out:
    first = True
    for filename in glob(f'{wd}vcfs_redacted/*.vcf'):
        if first:
            with open(filename, 'r') as inp:
                lines = inp.readlines()
            first = False
        else:
            with open(filename, 'r') as inp:
                lines = [l for l in inp.readlines() if not (l.startswith('#'))]
        out.write(''.join(lines))